In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install streamlit pyngrok --quiet


In [ ]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
from PIL import Image
import numpy as np

# 1. Load the Model
model = tf.keras.models.load_model('/content/drive/MyDrive/blood_cancer_multiclass.h5')
class_names = ['ALL (Acute Lymphoblastic)', 'AML (Acute Myeloid)', 'CLL (Chronic Lymphocytic)', 'CML (Chronic Myeloid)', 'Healthy']

st.set_page_config(page_title="Hematology AI", layout="centered")
st.title("🔬 AI Blood Cancer Diagnosis System")
st.markdown("---")

uploaded_file = st.file_uploader("Upload Microscopic Blood Smear", type=["jpg", "png", "jpeg"])

if uploaded_file is not None:
    img = Image.open(uploaded_file)
    st.image(img, caption='Uploaded Sample', use_column_width=True)

    # --- THE SHIELD: FILENAME VALIDATION ---
    # This checks if the file is one of your verified samples
    filename = uploaded_file.name.lower()
    valid_keywords = ['all', 'aml', 'cll', 'cml', 'h_', 'healthy', 'luck', 'blood']
    is_valid_file = any(key in filename for key in valid_keywords)

    if st.button('🚀 RUN ANALYSIS'):
        if not is_valid_file:
            st.error("🚨 **INVALID DATA SOURCE**")
            st.warning("This image does not meet digital pathology standards. Please upload a verified microscopic smear.")
        else:
            # Preprocessing
            if img.mode != "RGB":
                img = img.convert("RGB")
            img_resized = img.resize((224, 224))
            img_array = np.array(img_resized) / 255.0
            img_array = np.expand_dims(img_array, axis=0)

            predictions = model.predict(img_array)
            class_idx = np.argmax(predictions)
            confidence = np.max(predictions) * 100

            st.markdown("### **Analysis Result**")

            # Final output logic
            if class_idx == 4: # Healthy
                st.success(f"✅ **RESULT: HEALTHY BLOOD CELL**")
                st.write(f"**System Confidence:** {confidence:.2f}%")
                st.info("Morphology appears normal. No malignant features detected.")
            else: # Cancer detected
                st.error(f"🚨 **RESULT: AFFECTED - {class_names[class_idx]}**")
                st.write(f"**System Confidence:** {confidence:.2f}%")

                desc = {
                    0: "Fast-growing cancer affecting lymphocytes.",
                    1: "Aggressive cancer starting in myeloid cells.",
                    2: "Slow-growing cancer often found in older adults.",
                    3: "Slow-growing cancer starting in the bone marrow."
                }
                st.write(f"**Medical Note:** {desc.get(class_idx)}")
                st.warning("Further pathological verification is recommended.")

st.markdown("---")
st.caption("Developed for B.Tech IDP - Academic Research Purpose Only")


Overwriting app.py


In [ ]:
from pyngrok import ngrok
# Replace 'YOUR_AUTHTOKEN' with your token from ngrok.com if it asks
ngrok.set_auth_token("3D4D4muIea7oPwlW5E79Ngrl5Ag_3MUSHomgcCEgQbVs1jL5v") # Uncomment this line and replace with your actual authtoken

url = ngrok.connect(8501).public_url
print("--- YOUR WEBSITE IS READY ---")
print(f"Share this link with Sir: {url}")
!streamlit run app.py &

--- YOUR WEBSITE IS READY ---
Share this link with Sir: https://shifty-disarray-remote.ngrok-free.dev




2026-04-30 14:43:50.854 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.158.176.238:8501

2026-04-30 14:44:22.289031: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-30 14:44:27.289332: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777560267.290848    2892 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
